# Exportación del pipeline ML de conceptos


Este notebook reúne los procedimientos necesarios para exportar el modelo a ONNX y preparar su integración en un motor compatible, incluyendo la normalización Unicode, el procesamiento del texto, la tokenización, TF-IDF y el clasificador entrenado.


## 1. Configuración del entorno


In [18]:
%pip install -q onnx joblib numpy cmake ninja

Note: you may need to restart the kernel to use updated packages.


In [19]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import unicodedata
from pathlib import Path

import joblib
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper

In [20]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parents[1]
elif PROJECT_DIR.name == "ml":
    PROJECT_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name == "tfm":
    PROJECT_DIR = PROJECT_DIR / "ecuador-einvoice-classifier"

ML_DIR = PROJECT_DIR / "ml"
DEFAULT_MODEL = ML_DIR / "outputs/models/training/concept_classifier.joblib"
DEFAULT_OUTPUT = ML_DIR / "outputs/models/export"
BUILD_DIR = Path(
    os.environ.get("INVOICE_ONNX_BUILD_DIR", "/tmp/invoice-onnx-runtime")
).resolve()
EXPORT_DIR = BUILD_DIR / "export-support"
GENERATED_DIR = BUILD_DIR / "generated"

CUSTOM_DOMAIN = "ec.einvoice"
RUNTIME_VERSION = "1.31.1"
ORT_VERSION = RUNTIME_VERSION
ORT_COMMIT = "6af1a932e2286d5c58166085d24c68783033edee"
UTF8PROC_VERSION = "2.10.0"
UTF8PROC_COMMIT = "a1b99daa2a3393884220264c927a48ba1251a9c6"
EMSDK_VERSION = "4.0.23"

native_sources = {}

## 2. Procesamiento de texto


Se prepara el código que permitirá ejecutar dentro de ONNX el mismo tratamiento de las descripciones utilizado durante el entrenamiento.


In [21]:
native_sources["text_pipeline.h"] = r"""#pragma once

#include <cstdint>
#include <map>
#include <string>
#include <unordered_map>
#include <vector>

namespace invoice {

std::string PrepareText(const std::string& raw);

class TextPipeline {
 public:
  TextPipeline(const std::string& vocabulary, std::vector<double> idf);
  size_t FeatureCount() const { return idf_.size(); }
  std::map<uint32_t, double> Transform(const std::string& raw) const;

 private:
  std::unordered_map<std::string, uint32_t> vocabulary_;
  std::vector<double> idf_;
};

}  // namespace invoice
"""

### 2.1. Normalización y transformación de las descripciones


Se normalizan los caracteres Unicode y los espacios, se convierte el texto a minúsculas y se extraen palabras individuales y pares consecutivos, conservando los índices del vocabulario y los pesos IDF del modelo entrenado.


In [22]:
native_sources["text_pipeline.cc"] = r"""#include "text_pipeline.h"

#include <algorithm>
#include <cmath>
#include <cstdlib>
#include <iterator>
#include <stdexcept>
#include <utility>

#include "unicode_tables.h"
#include "utf8proc.h"

namespace invoice {
namespace {

template <size_t Count>
bool Contains(const invoice_unicode::Range (&ranges)[Count], uint32_t point) {
  auto found = std::upper_bound(
      std::begin(ranges), std::end(ranges), point,
      [](uint32_t value, const auto& range) { return value < range.first; });
  return found != std::begin(ranges) && point <= std::prev(found)->last;
}

std::vector<uint32_t> Decode(const std::string& text) {
  std::vector<uint32_t> result;
  for (size_t position = 0; position < text.size();) {
    utf8proc_int32_t point;
    auto consumed = utf8proc_iterate(
        reinterpret_cast<const uint8_t*>(text.data() + position),
        static_cast<utf8proc_ssize_t>(text.size() - position), &point);
    if (consumed <= 0) throw std::invalid_argument("INVALID_UTF8");
    result.push_back(static_cast<uint32_t>(point));
    position += static_cast<size_t>(consumed);
  }
  return result;
}

void AppendPoint(std::string& result, uint32_t point) {
  uint8_t bytes[4];
  auto length = utf8proc_encode_char(static_cast<int32_t>(point), bytes);
  if (length <= 0) throw std::invalid_argument("INVALID_UNICODE_POINT");
  result.append(reinterpret_cast<const char*>(bytes), static_cast<size_t>(length));
}

bool NeighborIsCased(const std::vector<uint32_t>& text, size_t index, int direction) {
  auto position = static_cast<int64_t>(index) + direction;
  while (position >= 0 && position < static_cast<int64_t>(text.size())) {
    auto point = text[static_cast<size_t>(position)];
    if (!Contains(invoice_unicode::ignorable, point)) {
      return Contains(invoice_unicode::cased, point);
    }
    position += direction;
  }
  return false;
}

}  // namespace

std::string PrepareText(const std::string& raw) {
  if (std::string(utf8proc_unicode_version()) != "16.0.0") {
    throw std::runtime_error("UNICODE_VERSION_MISMATCH");
  }
  uint8_t* normalized = nullptr;
  auto length = utf8proc_map(
      reinterpret_cast<const uint8_t*>(raw.data()),
      static_cast<utf8proc_ssize_t>(raw.size()), &normalized,
      static_cast<utf8proc_option_t>(UTF8PROC_STABLE | UTF8PROC_COMPAT | UTF8PROC_COMPOSE));
  if (length < 0) throw std::invalid_argument("INVALID_UTF8");
  std::string normalized_text(reinterpret_cast<char*>(normalized), static_cast<size_t>(length));
  std::free(normalized);

  std::vector<uint32_t> collapsed;
  bool space_pending = false;
  for (auto point : Decode(normalized_text)) {
    if (Contains(invoice_unicode::space, point)) {
      space_pending = !collapsed.empty();
    } else {
      if (space_pending) collapsed.push_back(0x20);
      collapsed.push_back(point);
      space_pending = false;
    }
  }

  std::string result;
  for (size_t index = 0; index < collapsed.size(); ++index) {
    auto point = collapsed[index];
    if (point == 0x3A3 && NeighborIsCased(collapsed, index, -1) &&
        !NeighborIsCased(collapsed, index, 1)) {
      AppendPoint(result, 0x3C2);
      continue;
    }
    auto found = std::lower_bound(
        std::begin(invoice_unicode::lowercase), std::end(invoice_unicode::lowercase), point,
        [](const auto& mapping, uint32_t value) { return mapping.point < value; });
    if (found != std::end(invoice_unicode::lowercase) && found->point == point) {
      for (uint32_t offset = 0; offset < found->length; ++offset) {
        AppendPoint(result, invoice_unicode::lowercase_points[found->offset + offset]);
      }
    } else {
      AppendPoint(result, point);
    }
  }
  return result;
}

TextPipeline::TextPipeline(const std::string& vocabulary, std::vector<double> idf)
    : idf_(std::move(idf)) {
  size_t start = 0;
  uint32_t index = 0;
  while (start < vocabulary.size()) {
    auto end = vocabulary.find('\n', start);
    if (end == std::string::npos) end = vocabulary.size();
    vocabulary_.emplace(vocabulary.substr(start, end - start), index++);
    start = end + 1;
  }
  if (vocabulary_.size() != idf_.size()) throw std::invalid_argument("VOCABULARY_IDF_MISMATCH");
}

std::map<uint32_t, double> TextPipeline::Transform(const std::string& raw) const {
  auto prepared = PrepareText(raw);
  std::vector<std::string> tokens;
  std::string current;
  for (auto point : Decode(prepared)) {
    if (Contains(invoice_unicode::word, point)) {
      AppendPoint(current, point);
    } else if (!current.empty()) {
      tokens.push_back(std::move(current));
      current.clear();
    }
  }
  if (!current.empty()) tokens.push_back(std::move(current));

  std::map<uint32_t, double> result;
  for (size_t index = 0; index < tokens.size(); ++index) {
    auto add = [&](const std::string& term) {
      auto found = vocabulary_.find(term);
      if (found != vocabulary_.end()) result[found->second] += 1.0;
    };
    add(tokens[index]);
    if (index + 1 < tokens.size()) add(tokens[index] + " " + tokens[index + 1]);
  }
  double norm = 0;
  for (auto& [index, value] : result) {
    value *= idf_[index];
    norm += value * value;
  }
  norm = std::sqrt(norm);
  if (norm != 0) {
    for (auto& [index, value] : result) value /= norm;
  }
  return result;
}

}  // namespace invoice
"""

### 2.2. Incorporación del procesamiento al modelo


Se define la operación de preparación del texto dentro de ONNX, conservando el orden y el estado de cada descripción para permitir el procesamiento de los demás conceptos, incluso cuando alguno carezca de información.


In [23]:
native_sources["custom_ops.cc"] = r"""#include "include/onnxruntime_extensions.h"
#include "onnxruntime_cxx_api.h"
#include "text_pipeline.h"

#include <algorithm>
#include <mutex>
#include <stdexcept>

namespace {

OrtStatus* Failure(const std::exception& error) {
  return Ort::GetApi().CreateStatus(ORT_FAIL, error.what());
}

size_t VectorLength(const Ort::ConstValue& input) {
  auto shape = input.GetTensorTypeAndShapeInfo().GetShape();
  if (shape.size() != 1 || shape[0] < 0) throw std::invalid_argument("EXPECTED_VECTOR");
  return static_cast<size_t>(shape[0]);
}

std::vector<double> ReadIdf(const OrtKernelInfo* info) {
  int is_constant = 0;
  auto input = Ort::ConstKernelInfo{info}.GetTensorConstantInput(1, &is_constant);
  if (!is_constant || !input) throw std::invalid_argument("IDF_MUST_BE_CONSTANT");
  auto count = input.GetTensorTypeAndShapeInfo().GetElementCount();
  auto values = input.GetTensorData<double>();
  return {values, values + count};
}

class TextKernel {
 public:
  explicit TextKernel(const OrtKernelInfo* info)
      : pipeline_(Ort::ConstKernelInfo{info}.GetAttribute<std::string>("vocabulary"), ReadIdf(info)) {}

  OrtStatus* ComputeV2(OrtKernelContext* raw_context) {
    try {
      Ort::KernelContext context(raw_context);
      auto descriptions = context.GetInput(0);
      auto present = context.GetInput(2);
      auto rows = VectorLength(descriptions);
      if (VectorLength(present) != rows) throw std::invalid_argument("DESCRIPTION_MASK_LENGTH");
      auto provided = present.GetTensorData<bool>();
      std::vector<int64_t> shape{static_cast<int64_t>(rows), static_cast<int64_t>(pipeline_.FeatureCount())};
      auto features = context.GetOutput(0, shape).GetTensorMutableData<double>();
      std::fill(features, features + rows * pipeline_.FeatureCount(), 0.0);
      auto statuses = context.GetOutput(1, {static_cast<int64_t>(rows)}).GetTensorMutableData<int64_t>();

      for (size_t row = 0; row < rows; ++row) {
        statuses[row] = 0;
        if (!provided[row]) {
          statuses[row] = 1;
          continue;
        }
        try {
          auto length = descriptions.GetStringTensorElementLength(row);
          std::string text(length, '\0');
          descriptions.GetStringTensorElement(length, row, text.data());
          for (auto [index, value] : pipeline_.Transform(text)) {
            features[row * pipeline_.FeatureCount() + index] = value;
          }
        } catch (const std::invalid_argument&) {
          statuses[row] = 2;
        }
      }
      return nullptr;
    } catch (const std::exception& error) {
      return Failure(error);
    }
  }

 private:
  invoice::TextPipeline pipeline_;
};

struct TextOp : Ort::CustomOpBase<TextOp, TextKernel, true> {
  const char* GetName() const { return "PythonTextTfidf"; }
  const char* GetExecutionProviderType() const { return "CPUExecutionProvider"; }
  size_t GetInputTypeCount() const { return 3; }
  ONNXTensorElementDataType GetInputType(size_t index) const {
    return index == 0 ? ONNX_TENSOR_ELEMENT_DATA_TYPE_STRING :
           index == 1 ? ONNX_TENSOR_ELEMENT_DATA_TYPE_DOUBLE : ONNX_TENSOR_ELEMENT_DATA_TYPE_BOOL;
  }
  size_t GetOutputTypeCount() const { return 2; }
  ONNXTensorElementDataType GetOutputType(size_t index) const {
    return index == 0 ? ONNX_TENSOR_ELEMENT_DATA_TYPE_DOUBLE : ONNX_TENSOR_ELEMENT_DATA_TYPE_INT64;
  }
  OrtStatus* CreateKernelV2(const OrtApi&, const OrtKernelInfo* info, void** kernel) const {
    try { *kernel = new TextKernel(info); return nullptr; }
    catch (const std::exception& error) { return Failure(error); }
  }
};

}  // namespace

extern "C" OrtStatus* ORT_API_CALL RegisterCustomOps(OrtSessionOptions* options, const OrtApiBase* api_base) {
  Ort::InitApi(api_base->GetApi(ORT_API_VERSION));
  try {
    static TextOp text_op;
    static Ort::CustomOpDomain domain("ec.einvoice");
    static std::once_flag once;
    std::call_once(once, [&] { domain.Add(&text_op); });
    Ort::ThrowOnError(Ort::GetApi().AddCustomOpDomain(options, domain));
    return nullptr;
  } catch (const std::exception& error) {
    return Failure(error);
  }
}
"""

### 2.3. Conexión con el motor de ejecución

Esta interfaz permite que el motor reconozca y ejecute la operación de texto definida en el bloque anterior.


In [24]:
native_sources["include/onnxruntime_extensions.h"] = r"""#pragma once
#include "onnxruntime_c_api.h"

#ifdef __cplusplus
extern "C" {
#endif
OrtStatus* ORT_API_CALL RegisterCustomOps(OrtSessionOptions* options, const OrtApiBase* api_base);
#ifdef __cplusplus
}
#endif
"""

### 2.4. Preparación de los archivos de compilación

Se indican los fuentes que deben compilarse y las carpetas temporales que utilizará el proceso.


In [25]:
native_sources["CMakeLists.txt"] = r"""cmake_minimum_required(VERSION 3.28)
project(InvoiceOnnxExtensions LANGUAGES C CXX)

if(NOT INVOICE_UTF8PROC_DIR OR NOT INVOICE_GENERATED_DIR)
  message(FATAL_ERROR "Set INVOICE_UTF8PROC_DIR and INVOICE_GENERATED_DIR")
endif()

add_library(ocos_operators STATIC text_pipeline.cc "${INVOICE_UTF8PROC_DIR}/utf8proc.c")
file(WRITE "${CMAKE_CURRENT_BINARY_DIR}/extension_anchor.cc" "int invoice_extension_anchor = 0;\n")
add_library(noexcep_operators STATIC "${CMAKE_CURRENT_BINARY_DIR}/extension_anchor.cc")
add_library(ortcustomops STATIC custom_ops.cc)

foreach(target ocos_operators noexcep_operators ortcustomops)
  target_compile_features(${target} PRIVATE cxx_std_17)
  target_compile_definitions(${target} PRIVATE UTF8PROC_STATIC ORT_API_MANUAL_INIT)
  target_include_directories(${target} PUBLIC
    "$<BUILD_INTERFACE:${CMAKE_CURRENT_SOURCE_DIR}>"
    "$<BUILD_INTERFACE:${CMAKE_CURRENT_SOURCE_DIR}/include>"
    "$<BUILD_INTERFACE:${INVOICE_UTF8PROC_DIR}>"
    "$<BUILD_INTERFACE:${INVOICE_GENERATED_DIR}>"
    "$<BUILD_INTERFACE:${ONNXRUNTIME_INCLUDE_DIR}/core/session>")
  set_target_properties(${target} PROPERTIES POSITION_INDEPENDENT_CODE ON)
endforeach()

target_link_libraries(ortcustomops PRIVATE ocos_operators noexcep_operators)
add_library(onnxruntime_extensions INTERFACE)
target_link_libraries(onnxruntime_extensions INTERFACE ortcustomops)
install(TARGETS onnxruntime_extensions EXPORT onnxruntimeTargets)
"""

### 2.5. Conservación del tratamiento Unicode

Se generan las tablas de caracteres, espacios y minúsculas de Unicode 16.0.0 utilizadas por Python. Esto permite conservar el tratamiento de tildes, letras y números del modelo entrenado.


In [26]:
def intervals(values: list[int]) -> list[tuple[int, int]]:
    """Compress a sorted set of code points into inclusive intervals."""
    result: list[tuple[int, int]] = []

    for value in values:
        if result and value == result[-1][1] + 1:
            result[-1] = (result[-1][0], value)
        else:
            result.append((value, value))

    return result


def generate_unicode_header(destination: Path) -> dict[str, int | str]:
    """Freeze Python's lower, word, whitespace and casing context tables."""
    if unicodedata.unidata_version != "16.0.0":
        raise RuntimeError("This model requires Python Unicode 16.0.0.")

    properties: dict[str, list[int]] = {
        "word": [],
        "space": [],
        "cased": [],
        "ignorable": [],
    }
    mappings: list[tuple[int, int, int]] = []
    lowercase_points: list[int] = []

    for point in range(0x110000):
        character = chr(point)
        lowered = character.lower()

        if lowered != character:
            mappings.append((point, len(lowercase_points), len(lowered)))
            lowercase_points.extend(map(ord, lowered))

        cased = (
            character.islower()
            or character.isupper()
            or unicodedata.category(character) == "Lt"
        )

        if character.isalnum() or character == "_":
            properties["word"].append(point)

        if character.isspace():
            properties["space"].append(point)

        if cased:
            properties["cased"].append(point)

        if cased:
            ignorable = ("." + character + "Σ").lower().endswith("σ")
        else:
            ignorable = ("AΣ" + character + "A").lower()[1] == "σ"
        if ignorable:
            properties["ignorable"].append(point)

    lines = [
        "// Generated by generate_unicode.py; Python Unicode 16.0.0.",
        "#pragma once",
        "#include <cstdint>",
        "namespace invoice_unicode {",
        "struct Range { uint32_t first; uint32_t last; };",
        "struct Lower { uint32_t point; uint32_t offset; uint32_t length; };",
    ]

    for name, values in properties.items():
        lines.append(f"inline constexpr Range {name}[] = {{")
        lines.extend(
            f"{{0x{first:X}, 0x{last:X}}}," for first, last in intervals(values)
        )
        lines.append("};")

    lines.append("inline constexpr Lower lowercase[] = {")
    lines.extend(
        f"{{0x{point:X}, {offset}, {length}}}," for point, offset, length in mappings
    )
    lines.append("};")
    lines.append("inline constexpr uint32_t lowercase_points[] = {")
    lines.extend(f"0x{point:X}," for point in lowercase_points)
    lines.extend(["};", "}  // namespace invoice_unicode", ""])

    destination.parent.mkdir(parents=True, exist_ok=True)
    content = "\n".join(lines)

    if not destination.exists() or destination.read_text(encoding="utf-8") != content:
        destination.write_text(content, encoding="utf-8")

    return {"unicode": unicodedata.unidata_version, "lowercase_mappings": len(mappings)}

### 2.6. Caché de compilación


Se generan los archivos de los bloques anteriores en la caché de compilación, que contiene los recursos necesarios para construir el motor.


In [27]:
for name, source in native_sources.items():
    destination = EXPORT_DIR / "native" / name
    destination.parent.mkdir(parents=True, exist_ok=True)

    if not destination.exists() or destination.read_text(encoding="utf-8") != source:
        destination.write_text(source, encoding="utf-8")

unicode_tables = generate_unicode_header(GENERATED_DIR / "unicode_tables.h")

print(f"Soporte temporal: {EXPORT_DIR}")
print(f"Unicode: {unicode_tables['unicode']}")

Soporte temporal: /tmp/invoice-onnx-runtime/export-support
Unicode: 16.0.0


## 3. Construcción del pipeline ONNX


Se define el proceso que transforma las descripciones originales en categorías y probabilidades, incorporando durante la exportación el vocabulario, los pesos IDF, los coeficientes, los interceptos y el orden de las clases del modelo entrenado.


In [28]:
def runtime_signature() -> str:
    """Identificar el soporte nativo y las tablas que deben compartir modelo y motor."""
    digest = hashlib.sha256(b"ec.einvoice:1;unicode:16.0.0;utf8proc:2.10.0\n")

    for name in sorted(native_sources):
        digest.update(name.encode() + b"\0" + native_sources[name].encode("utf-8"))

    digest.update((GENERATED_DIR / "unicode_tables.h").read_bytes())

    return digest.hexdigest()


def tensor_info(name: str, dtype: int, shape: list) -> onnx.ValueInfoProto:
    """Declarar una entrada o salida tensorial del contrato exportado."""
    return helper.make_tensor_value_info(name, dtype, shape)


def build_graph(model: object) -> onnx.ModelProto:
    """Incluir preparación, TF-IDF y clasificación en un único grafo ONNX."""
    vectorizer = model.named_steps["tfidf"]
    classifier = model.named_steps["classifier"]
    expected = {
        "ngram_range": (1, 2),
        "token_pattern": r"(?u)\b\w+\b",
        "lowercase": True,
        "strip_accents": None,
        "norm": "l2",
        "smooth_idf": True,
        "sublinear_tf": False,
        "analyzer": "word",
        "binary": False,
    }

    for parameter, value in expected.items():
        if vectorizer.get_params()[parameter] != value:
            raise ValueError(f"El modelo requiere otro preprocesamiento: {parameter}")

    if unicodedata.unidata_version != "16.0.0":
        raise RuntimeError(
            "La exportación requiere las tablas Unicode 16.0.0 del modelo."
        )

    classes = classifier.classes_.tolist()
    class_count = len(classes)
    vocabulary = [""] * len(vectorizer.vocabulary_)

    for term, index in vectorizer.vocabulary_.items():
        vocabulary[index] = term

    initializers = [
        numpy_helper.from_array(np.asarray(vectorizer.idf_, dtype=np.float64), "idf"),
        numpy_helper.from_array(
            np.asarray(classifier.coef_.T, dtype=np.float64), "coefficients"
        ),
        numpy_helper.from_array(
            np.asarray(classifier.intercept_, dtype=np.float64), "intercepts"
        ),
        numpy_helper.from_array(np.array(0, dtype=np.int64), "zero_status"),
        numpy_helper.from_array(np.array(-1, dtype=np.int64), "missing_category"),
        numpy_helper.from_array(
            np.array(class_count, dtype=np.int64), "missing_label_index"
        ),
        numpy_helper.from_array(np.array(0.0, dtype=np.float64), "zero_probability"),
        numpy_helper.from_array(np.array([1], dtype=np.int64), "column_axis"),
        helper.make_tensor(
            "class_labels", TensorProto.STRING, [class_count + 1], classes + [""]
        ),
    ]

    nodes = [
        helper.make_node(
            "PythonTextTfidf",
            ["descriptions", "idf", "description_present"],
            ["features", "concept_statuses"],
            domain=CUSTOM_DOMAIN,
            name="prepare_text_and_tfidf",
            vocabulary="\n".join(vocabulary),
        ),
        helper.make_node(
            "MatMul",
            ["features", "coefficients"],
            ["linear_scores"],
            name="classifier_weights",
        ),
        helper.make_node(
            "Add",
            ["linear_scores", "intercepts"],
            ["scores"],
            name="classifier_intercepts",
        ),
        helper.make_node(
            "Softmax",
            ["scores"],
            ["predicted_probabilities"],
            axis=1,
            name="class_probabilities",
        ),
        helper.make_node(
            "ArgMax",
            ["predicted_probabilities"],
            ["predicted_indices"],
            axis=1,
            keepdims=0,
            name="select_concept_category",
        ),
        helper.make_node(
            "Equal", ["concept_statuses", "zero_status"], ["valid_predictions"]
        ),
        helper.make_node(
            "Where",
            ["valid_predictions", "predicted_indices", "missing_category"],
            ["category_indices"],
        ),
        helper.make_node(
            "Where",
            ["valid_predictions", "predicted_indices", "missing_label_index"],
            ["label_indices"],
        ),
        helper.make_node(
            "Gather", ["class_labels", "label_indices"], ["categories"], axis=0
        ),
        helper.make_node(
            "Unsqueeze", ["valid_predictions", "column_axis"], ["valid_rows"]
        ),
        helper.make_node(
            "Where",
            ["valid_rows", "predicted_probabilities", "zero_probability"],
            ["probabilities"],
        ),
    ]

    inputs = [
        tensor_info("descriptions", TensorProto.STRING, ["concepts"]),
        tensor_info("description_present", TensorProto.BOOL, ["concepts"]),
    ]

    outputs = [
        tensor_info("categories", TensorProto.STRING, ["concepts"]),
        tensor_info("category_indices", TensorProto.INT64, ["concepts"]),
        tensor_info("probabilities", TensorProto.DOUBLE, ["concepts", class_count]),
        tensor_info("concept_statuses", TensorProto.INT64, ["concepts"]),
    ]

    graph = helper.make_graph(
        nodes, "concept_classification_pipeline", inputs, outputs, initializers
    )

    result = helper.make_model(
        graph,
        producer_name="ecuador-einvoice-classifier",
        opset_imports=[
            helper.make_opsetid("", 18),
            helper.make_opsetid(CUSTOM_DOMAIN, 1),
        ],
    )
    result.ir_version = 10
    helper.set_model_props(
        result,
        {
            "classes": json.dumps(classes, ensure_ascii=False),
            "unicode_version": "16.0.0",
            "runtime_version": RUNTIME_VERSION,
            "runtime_signature": runtime_signature(),
            "contract": "DN-0004 revised 2026-11-03",
            "source": "trained-joblib-without-refit",
        },
    )

    onnx.checker.check_model(result)

    return result

### 3.1. Preparación del guardado


Se establecen las comprobaciones de compatibilidad del motor y el almacenamiento del modelo con sus recursos, preparando el nuevo paquete antes de reemplazar la versión anterior, que se conservará si ocurre un error durante la publicación.


In [29]:
def export_pipeline(
    model_path: Path = DEFAULT_MODEL,
    output_dir: Path = DEFAULT_OUTPUT,
    *,
    runtime_dir: Path | None = None,
) -> dict:
    """Guardar ONNX y empaquetar un motor compatible cuando esté disponible."""
    model_path = Path(model_path).resolve()
    output_dir = Path(output_dir).resolve()

    model = joblib.load(model_path)
    exported = build_graph(model)
    model_bytes = exported.SerializeToString()
    candidates = {"concept_pipeline.onnx": model_bytes}

    runtime = None
    runtime_source = (
        Path(runtime_dir).resolve() if runtime_dir else output_dir / "runtime"
    )

    if (runtime_source / "runtime.json").exists():
        runtime = json.loads((runtime_source / "runtime.json").read_text())
        if (
            runtime["version"] != RUNTIME_VERSION
            or runtime["signature"] != runtime_signature()
        ):
            raise RuntimeError(
                "El motor no es compatible con esta versión del pipeline."
            )

        for name, digest in runtime["files"].items():
            asset = runtime_source / name
            contents = asset.read_bytes()

            if hashlib.sha256(contents).hexdigest() != digest:
                raise RuntimeError(f"El recurso de ejecución está modificado: {name}")

            candidates[f"runtime/{name}"] = contents

        candidates["runtime/runtime.json"] = (
            runtime_source / "runtime.json"
        ).read_bytes()

    manifest = {
        "model": "concept_pipeline.onnx",
        "sha256": hashlib.sha256(model_bytes).hexdigest(),
        "source_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
        "classes": model.named_steps["classifier"].classes_.tolist(),
        "features": len(model.named_steps["tfidf"].vocabulary_),
        "unicode_version": unicodedata.unidata_version,
        "python_version": platform.python_version(),
        "runtime_signature": runtime_signature(),
        "runtime_packaged": runtime is not None,
        "runtime": runtime
        or {"version": RUNTIME_VERSION, "requires_invoice_text_extension": True},
        "inputs": {
            "descriptions": "raw string vector in original concept order",
            "description_present": "boolean vector: source field exists",
        },
        "outputs": {
            "categories": "category labels; empty for failed concepts",
            "category_indices": "indices in classes; -1 for failed concepts",
            "probabilities": "float64 matrix; columns follow classes; failed rows are zero",
            "concept_statuses": {
                "0": "ok",
                "1": "missing_description",
                "2": "invalid_unicode",
            },
        },
    }
    candidates["manifest.json"] = (
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n"
    ).encode("utf-8")

    # Validar y escribir la versión candidata antes de sustituir el paquete vigente.
    output_dir.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory(
        prefix=f".{output_dir.name}-", dir=output_dir.parent
    ) as temporary:
        temporary_dir = Path(temporary)
        staged_dir = temporary_dir / "bundle"

        if output_dir.exists():
            shutil.copytree(output_dir, staged_dir, symlinks=True)
        else:
            staged_dir.mkdir()

        for name, contents in candidates.items():
            target = staged_dir / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(contents)

        previous_dir = temporary_dir / "previous"

        if output_dir.exists():
            os.replace(output_dir, previous_dir)
        try:
            os.replace(staged_dir, output_dir)
        except BaseException:
            if previous_dir.exists():
                os.replace(previous_dir, output_dir)
            raise

    return manifest

## 4. Preparación del motor para el navegador


El motor WebAssembly permitirá ejecutar el modelo ONNX directamente en la CPU del navegador, utilizando una versión compilada previamente que se distribuirá junto con el modelo para realizar predicciones locales. Para su construcción, se establecen las dependencias necesarias y se utiliza una carpeta temporal para las descargas y los archivos de trabajo.


In [30]:
def run(
    command: list[str], *, cwd: Path | None = None, env: dict | None = None
) -> None:
    """Ejecutar una dependencia de compilación y propagar sus fallos."""
    log_path = BUILD_DIR / "runtime-build.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open("ab") as log_file:
        try:
            subprocess.run(
                command,
                cwd=cwd,
                env=env,
                check=True,
                stdout=log_file,
                stderr=subprocess.STDOUT,
            )
        except subprocess.CalledProcessError as error:
            raise RuntimeError(
                f"Falló un paso de compilación. Revisa {log_path}"
            ) from error


def pinned_source(destination: Path, repository: str, tag: str, commit: str) -> None:
    """Obtener una versión fija sin sobrescribir otros árboles existentes."""
    if not destination.exists():
        run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                tag,
                repository,
                str(destination),
            ]
        )

    observed = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=destination, text=True
    ).strip()

    if observed != commit:
        raise RuntimeError(
            f"La dependencia {destination.name} no corresponde a la versión fijada."
        )

### 4.1. Compilación del motor


Se configura la compilación del soporte de procesamiento de texto y del motor compatible, generando el binario WebAssembly, su cargador y la información de versiones necesaria para garantizar su funcionamiento conjunto.


In [31]:
def build_runtime(work_dir: Path, output_dir: Path, jobs: int = 4) -> dict:
    """Construir y empaquetar un motor SIMD de un hilo compatible con el modelo."""
    work_dir = Path(work_dir).resolve()
    output_dir = Path(output_dir).resolve()
    work_dir.mkdir(parents=True, exist_ok=True)

    ort_dir = work_dir / "onnxruntime"
    utf8proc_dir = work_dir / "utf8proc"

    pinned_source(
        ort_dir,
        "https://github.com/microsoft/onnxruntime.git",
        f"v{ORT_VERSION}",
        ORT_COMMIT,
    )
    pinned_source(
        utf8proc_dir,
        "https://github.com/JuliaStrings/utf8proc.git",
        f"v{UTF8PROC_VERSION}",
        UTF8PROC_COMMIT,
    )

    run(
        [
            "git",
            "submodule",
            "update",
            "--init",
            "--depth",
            "1",
            "cmake/external/emsdk",
        ],
        cwd=ort_dir,
    )

    emsdk_dir = ort_dir / "cmake/external/emsdk"
    emsdk = emsdk_dir / "emsdk"

    if not (emsdk_dir / "upstream/emscripten/emcc").exists():
        run([str(emsdk), "install", EMSDK_VERSION])

    run([str(emsdk), "activate", EMSDK_VERSION])

    environment = os.environ.copy()
    environment["PATH"] = (
        str(Path(sys.executable).parent) + os.pathsep + environment["PATH"]
    )
    environment["EM_CACHE"] = str(work_dir / "emscripten-cache")

    emitted = subprocess.check_output(
        [
            "bash",
            "-c",
            'source "$1" >/dev/null 2>&1; env -0',
            "bash",
            str(emsdk_dir / "emsdk_env.sh"),
        ],
        env=environment,
    )

    for item in emitted.split(b"\0"):
        if b"=" in item:
            key, value = item.split(b"=", 1)
            environment[os.fsdecode(key)] = os.fsdecode(value)

    cmake = shutil.which("cmake", path=environment["PATH"])
    ninja = shutil.which("ninja", path=environment["PATH"])

    if not cmake or not ninja:
        raise RuntimeError("Instala cmake y ninja en el entorno de exportación.")

    generated_dir = work_dir / "generated"

    generate_unicode_header(generated_dir / "unicode_tables.h")

    operators_file = generated_dir / "required_operators.config"
    operators_file.write_text(
        "ai.onnx;18;Add,ArgMax,Equal,Gather,Gemm,MatMul,Softmax,Unsqueeze,Where\n"
    )

    print(
        "Compilando ONNX Runtime SIMD de un hilo con las operaciones del pipeline...",
        flush=True,
    )

    run(
        [
            sys.executable,
            str(ort_dir / "tools/ci_build/build.py"),
            "--build_dir",
            str(work_dir / "wasm-build"),
            "--config",
            "Release",
            "--update",
            "--build",
            "--build_wasm",
            "--enable_wasm_simd",
            "--skip_tests",
            "--disable_rtti",
            "--disable_contrib_ops",
            "--disable_ml_ops",
            "--parallel",
            str(jobs),
            "--cmake_generator",
            "Ninja",
            "--cmake_path",
            cmake,
            "--include_ops_by_config",
            str(operators_file),
            "--cmake_extra_defines",
            "onnxruntime_USE_EXTENSIONS=ON",
            "onnxruntime_EXTENSIONS_OVERRIDDEN=ON",
            f"onnxruntime_EXTENSIONS_PATH={EXPORT_DIR / 'native'}",
            f"INVOICE_UTF8PROC_DIR={utf8proc_dir}",
            f"INVOICE_GENERATED_DIR={generated_dir}",
            f"CMAKE_MAKE_PROGRAM={ninja}",
            "CMAKE_POLICY_VERSION_MINIMUM=3.5",
            "onnxruntime_BUILD_UNIT_TESTS=OFF",
        ],
        cwd=ort_dir,
        env=environment,
    )

    release_dir = work_dir / "wasm-build/Release"

    binaries = sorted(release_dir.glob("ort-wasm*.wasm"))
    binaries = [
        path
        for path in binaries
        if not any(
            part in path.name for part in ["jsep", "jspi", "asyncify", "training"]
        )
    ]

    if len(binaries) != 1:
        raise RuntimeError("No se encontró un único binario WebAssembly de producción.")

    wasm_path = binaries[0]
    module_path = wasm_path.with_suffix(".mjs")

    if not module_path.exists():
        raise FileNotFoundError(module_path)

    package_dir = work_dir / "javascript"
    package = package_dir / "node_modules/onnxruntime-web"

    if not package.exists():
        run(
            [
                "npm",
                "install",
                "--prefix",
                str(package_dir),
                "--no-audit",
                "--no-fund",
                "--ignore-scripts",
                f"onnxruntime-web@{ORT_VERSION}",
            ]
        )

    version = json.loads((package / "package.json").read_text())["version"]

    if version != ORT_VERSION:
        raise RuntimeError(
            "El cargador JavaScript y el motor WebAssembly tienen versiones distintas."
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    assets = {
        wasm_path.name: wasm_path,
        module_path.name: module_path,
        "ort.wasm.min.mjs": package / "dist/ort.wasm.min.mjs",
        "LICENSE.onnxruntime": ort_dir / "LICENSE",
        "ThirdPartyNotices.onnxruntime.txt": ort_dir / "ThirdPartyNotices.txt",
        "LICENSE.utf8proc.md": utf8proc_dir / "LICENSE.md",
    }

    for name, path in assets.items():
        shutil.copy2(path, output_dir / name)

    manifest = {
        "version": ORT_VERSION,
        "source_commit": ORT_COMMIT,
        "emscripten": EMSDK_VERSION,
        "signature": runtime_signature(),
        "unicode_version": "16.0.0",
        "utf8proc_version": UTF8PROC_VERSION,
        "threads": 1,
        "simd": True,
        "loader": "ort.wasm.min.mjs",
        "module": module_path.name,
        "wasm": wasm_path.name,
        "files": {
            name: hashlib.sha256(path.read_bytes()).hexdigest()
            for name, path in assets.items()
        },
    }
    (output_dir / "runtime.json").write_text(json.dumps(manifest, indent=2) + "\n")

    return manifest

### 4.2. Reutilización o construcción del motor


Se comprueba si existe un motor compatible con el soporte de texto preparado para reutilizarlo o, en caso contrario, compilar uno nuevo en la carpeta temporal.


In [32]:
runtime_dir = BUILD_DIR / "runtime-package"
runtime_candidates = [runtime_dir, DEFAULT_OUTPUT / "runtime"]

for candidate in runtime_candidates:
    descriptor = candidate / "runtime.json"

    if descriptor.exists():
        candidate_manifest = json.loads(descriptor.read_text())

        if candidate_manifest.get("signature") == runtime_signature():
            runtime_dir = candidate
            break
else:
    runtime_manifest = build_runtime(BUILD_DIR, runtime_dir)

print(f"Motor compatible: {runtime_dir}")

Motor compatible: /tmp/invoice-onnx-runtime/runtime-package


## 5. Exportación y guardado


Se carga el modelo entrenado y se exporta el pipeline ML, guardándolo junto con el motor compatible y su manifiesto, mientras que la salida presenta las rutas del paquete, las categorías y las características del modelo.


In [33]:
manifest = export_pipeline(DEFAULT_MODEL, DEFAULT_OUTPUT, runtime_dir=runtime_dir)

print(f"Modelo exportado: {DEFAULT_OUTPUT / manifest['model']}")
print(f"Categorías: {len(manifest['classes'])}")
print(f"Características TF-IDF: {manifest['features']:,}")
print(f"Unicode: {manifest['unicode_version']}")
print(f"Motor empaquetado: {manifest['runtime_packaged']}")
print(f"Contrato de despliegue: {DEFAULT_OUTPUT / 'manifest.json'}")

Modelo exportado: /home/joletuar/personal/unir-workspace/tfm/ecuador-einvoice-classifier/ml/outputs/models/export/concept_pipeline.onnx
Categorías: 7
Características TF-IDF: 26,780
Unicode: 16.0.0
Motor empaquetado: True
Contrato de despliegue: /home/joletuar/personal/unir-workspace/tfm/ecuador-einvoice-classifier/ml/outputs/models/export/manifest.json


In [34]:
source_pipeline_bytes = DEFAULT_MODEL.stat().st_size
print(
    f"Pipeline sklearn de origen: {source_pipeline_bytes:,} bytes "
    f"({source_pipeline_bytes / 1024**2:.4f} MiB)"
)

package_bytes = 0
runtime_bytes = 0
packaged_runtime = DEFAULT_OUTPUT / "runtime"

for artifact_path in sorted(DEFAULT_OUTPUT.rglob("*")):
    if not artifact_path.is_file():
        continue

    artifact_bytes = artifact_path.stat().st_size
    artifact_name = artifact_path.relative_to(DEFAULT_OUTPUT).as_posix()
    package_bytes += artifact_bytes

    if artifact_path.is_relative_to(packaged_runtime):
        runtime_bytes += artifact_bytes

    print(f"{artifact_name}: {artifact_bytes:,} bytes ({artifact_bytes / 1024**2:.4f} MiB)")

print(f"Subtotal del runtime: {runtime_bytes:,} bytes ({runtime_bytes / 1024**2:.4f} MiB)")
print(f"Total del paquete exportado: {package_bytes:,} bytes ({package_bytes / 1024**2:.4f} MiB)")


Pipeline sklearn de origen: 2,120,052 bytes (2.0218 MiB)
concept_pipeline.onnx: 1,986,414 bytes (1.8944 MiB)
manifest.json: 2,107 bytes (0.0020 MiB)
runtime/LICENSE.onnxruntime: 1,073 bytes (0.0010 MiB)
runtime/LICENSE.utf8proc.md: 5,125 bytes (0.0049 MiB)
runtime/ThirdPartyNotices.onnxruntime.txt: 338,088 bytes (0.3224 MiB)
runtime/ort-wasm-simd.mjs: 26,331 bytes (0.0251 MiB)
runtime/ort-wasm-simd.wasm: 9,167,891 bytes (8.7432 MiB)
runtime/ort.wasm.min.mjs: 53,725 bytes (0.0512 MiB)
runtime/runtime.json: 985 bytes (0.0009 MiB)
Subtotal del runtime: 9,593,218 bytes (9.1488 MiB)
Total del paquete exportado: 11,581,739 bytes (11.0452 MiB)


### 5.1. Archivos generados y uso del modelo

El paquete se guarda en `ml/outputs/models/export`:

| Archivo                 | Contenido                                                            |
| ----------------------- | -------------------------------------------------------------------- |
| `concept_pipeline.onnx` | Preparación del texto, TF-IDF y clasificador de conceptos.           |
| `manifest.json`         | Clases, entradas, salidas y versión del motor compatible.            |
| `runtime/`              | Motor WebAssembly, cargador, versiones y avisos de las dependencias. |


### 5.2. Formato de `manifest.json`

El archivo `manifest.json` contiene la información necesaria para identificar el modelo exportado, sus categorías y los recursos requeridos para ejecutarlo.

| Campo               | Tipo            | Descripción                                                                                             |
| ------------------- | --------------- | ------------------------------------------------------------------------------------------------------- |
| `model`             | `string`        | Nombre del archivo ONNX que se cargará.                                                                 |
| `sha256`            | `string`        | Huella SHA-256 del modelo ONNX exportado.                                                               |
| `source_sha256`     | `string`        | Huella SHA-256 del modelo entrenado original.                                                           |
| `classes`           | `array<string>` | Categorías ordenadas según los índices de clase y las columnas de probabilidades.                       |
| `features`          | `integer`       | Cantidad de características TF-IDF del modelo.                                                          |
| `unicode_version`   | `string`        | Versión Unicode utilizada en la preparación del texto.                                                  |
| `python_version`    | `string`        | Versión de Python utilizada durante la exportación.                                                     |
| `runtime_signature` | `string`        | Firma del procesamiento de texto que debe coincidir con `runtime.signature`.                            |
| `runtime_packaged`  | `boolean`       | Indica si el paquete contiene el motor compatible.                                                      |
| `runtime`           | `object`        | Información del motor incluido, correspondiente a `runtime/runtime.json` y descrita en el apartado 5.4. |
| `inputs`            | `object`        | Define las entradas `descriptions` y `description_present`.                                             |
| `outputs`           | `object`        | Define las salidas del modelo y los códigos de `concept_statuses`.                                      |

### 5.3. Entradas y salidas del modelo

Los objetos `inputs` y `outputs` de `manifest.json` describen los datos que recibe y devuelve el modelo ONNX, utilizando vectores y matrices en lugar de respuestas JSON de factura. Cada llamada procesa `N` conceptos y conserva su orden original, de modo que cada resultado corresponde a la misma posición de entrada.

Entradas

| Campo                 | Tipo        | Descripción                                                                                   |
| --------------------- | ----------- | --------------------------------------------------------------------------------------------- |
| `descriptions`        | `string[N]` | Descripciones originales de los conceptos, sin procesamiento previo en JavaScript.            |
| `description_present` | `bool[N]`   | Indica si el campo de descripción está presente, diferenciando su ausencia de un texto vacío. |

Salidas

| Campo              | Tipo            | Descripción                                                                                                      |
| ------------------ | --------------- | ---------------------------------------------------------------------------------------------------------------- |
| `categories`       | `string[N]`     | Categoría asignada a cada concepto o cadena vacía si falla su procesamiento.                                     |
| `category_indices` | `int64[N]`      | Índice de la categoría en `classes` o `-1` cuando ocurre un fallo.                                               |
| `probabilities`    | `float64[N, 7]` | Probabilidades de las siete categorías, ordenadas según `classes`, con valores cero para los conceptos fallidos. |
| `concept_statuses` | `int64[N]`      | Código que indica el resultado del procesamiento de cada concepto.                                               |

El campo `concept_statuses` utiliza los siguientes códigos:

| Código | Estado                | Descripción                                                             |
| ------ | --------------------- | ----------------------------------------------------------------------- |
| `0`    | `ok`                  | Concepto procesado correctamente.                                       |
| `1`    | `missing_description` | El concepto no contiene el campo de descripción.                        |
| `2`    | `invalid_unicode`     | Se detectó un error Unicode durante el procesamiento de la descripción. |


### 5.4. Formato de `runtime/runtime.json`

El archivo `runtime/runtime.json` contiene la información del motor WebAssembly y los recursos necesarios para ejecutar el modelo ONNX, permitiendo comprobar su compatibilidad e integridad. 

| Campo              | Tipo      | Descripción                                                                                                    |
| ------------------ | --------- | -------------------------------------------------------------------------------------------------------------- |
| `version`          | `string`  | Versión de ONNX Runtime utilizada por el motor y su cargador.                                                  |
| `source_commit`    | `string`  | Revisión del código fuente utilizada para compilar ONNX Runtime.                                               |
| `emscripten`       | `string`  | Versión de Emscripten utilizada en la compilación de WebAssembly.                                              |
| `signature`        | `string`  | Firma del procesamiento de texto que debe coincidir con `manifest.runtime_signature`.                          |
| `unicode_version`  | `string`  | Versión Unicode utilizada por el motor.                                                                        |
| `utf8proc_version` | `string`  | Versión de la biblioteca utilizada para la normalización NFKC.                                                 |
| `threads`          | `integer` | Número de hilos configurados para la ejecución.                                                                |
| `simd`             | `boolean` | Indica si el motor incluye optimizaciones SIMD.                                                                |
| `loader`           | `string`  | Nombre del archivo JavaScript que carga ONNX Runtime.                                                          |
| `module`           | `string`  | Nombre del módulo JavaScript que inicializa WebAssembly.                                                       |
| `wasm`             | `string`  | Nombre del archivo binario WebAssembly.                                                                        |
| `files`            | `object`  | Relación de los archivos distribuidos y sus respectivas huellas SHA-256, incluidos los avisos de dependencias. |
